In [ ]:
import xarray as xr
import matplotlib.pylab as plt
import xarray_regrid.methods.conservative  # pylint: disable=unused-import
from functools import partial
import xskillscore as xs
import pint_xarray
from dask.diagnostics import ProgressBar
import json

from pism_terra.filtering import importance_sampling
from pism_terra.processing import preprocess_netcdf as preprocess

data_dir = "/Volumes/LHS800DATA"

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
delta_coder = xr.coders.CFTimedeltaCoder()
obs = xr.open_dataset(f"{data_dir}/2026_04_kitp_debm_calib/kitp/input/v3/HIRHAM5-ERA5_YMM_1990_2019_v3.nc", engine="netcdf4",
                     decode_times=False,
                     decode_timedelta=False, chunks=None).drop_dims("nv", errors="ignore")

obs = obs.pint.quantify()
for v in ["surface_melt_flux", "surface_runoff_flux", "climatic_mass_balance"]:
    obs[v] = obs[v].pint.to("kg m^-2 yr^-1")
obs = obs.pint.dequantify()

debm = xr.open_mfdataset(f"{data_dir}/2026_04_kitp_debm_calib/output/spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
                         preprocess=partial(preprocess,
                                            uq_regexp=None,
                                            exp_regexp="uq_(.+?)_"),
                        engine="netcdf4",
                        join="outer",
                        compat="no_conflicts",
                        parallel=True,
                        chunks="auto",
                        decode_times=False,
                        decode_timedelta=False).drop_dims("nv", errors="ignore")

debm_uq_vars = ['surface.debm_simple.c1', 'surface.debm_simple.c2','surface.debm_simple.air_temp_all_precip_as_snow', 'surface.debm_simple.air_temp_all_precip_as_rain', 'surface.debm_simple.refreeze']
debm_uq_df = debm.pism_config.to_series().apply(json.loads).apply(pd.Series)[debm_uq_vars]
debm["time"] = obs["time"]

obs = obs.regrid.conservative(debm.drop_vars("pism_config")).squeeze()
obs["surface_melt_flux_error"] = xr.where(obs["surface_melt_flux"] > 0, 0.10 * obs["surface_melt_flux"], 1e-4)

pdd = xr.open_mfdataset(f"{data_dir}/2026_04_kitp_pdd_calib/output/spatial/clipped_spatial_g4800m_id_HIRHAM5-ERA5_YMM_1990_2019_uq_*.nc",
                         preprocess=partial(preprocess,
                                            uq_regexp=None,
                                            exp_regexp="uq_(.+?)_"),
                        engine="netcdf4",
                        join="outer",
                        compat="no_conflicts",
                        parallel=True,
                        chunks="auto",
                        decode_times=False,
                        decode_timedelta=False).drop_dims("nv", errors="ignore")

pdd_uq_vars = ['surface.pdd.factor_ice', 'surface.pdd.factor_snow','surface.pdd.refreeze']
pdd_uq_df = pdd.pism_config.to_series().apply(json.loads).apply(pd.Series)[pdd_uq_vars]
pdd["time"] = obs["time"]

# pdd_rmse = xs.rmse(pdd_melt, obs_melt, dim=["y", "x"], skipna=True).compute()

In [ ]:
with ProgressBar():
    debm_filtered = importance_sampling(debm, obs, 
                    sim_var="surface_melt_flux",
                    obs_mean_var="surface_melt_flux", 
                    obs_std_var="surface_melt_flux_error",
                    sum_dims=['time', 'x', 'y'],
                    fudge_factor=3
                                       )

    pdd_filtered = importance_sampling(pdd, obs, 
                    sim_var="surface_melt_flux",
                    obs_mean_var="surface_melt_flux", 
                    obs_std_var="surface_melt_flux_error",
                    sum_dims=['time', 'x', 'y'],
                    fudge_factor=3
                                        )



In [ ]:
pdd_uq_df.head()

In [ ]:
import numpy as np
sampled_ids = pdd_filtered.exp_id_sampled.values
counts = pd.Series(sampled_ids).value_counts()
                                                                                                                                                                                     
# Reindex config_df to the sampled IDs and plot histograms                                                                                                                         
sampled_configs = pdd_uq_df.loc[counts.index].reindex(counts.index)                                                                                                                
                                                                                                                                                                                     
fig, axes = plt.subplots(1, 3, figsize=(12, 4))           
for ax, var in zip(axes, pdd_uq_vars):                                                                                                                                             
        # Repeat each parameter value by its sample count                                                                                                                              
          values = np.repeat(pdd_uq_df[var].values,                                                                                                                                      
                         counts.reindex(pdd_uq_df.index, fill_value=0).values.astype(int))                                                                                           
          ax.hist(values, bins=15)                                                                                                                                                       
          ax.set_xlabel(var)                                    
          ax.set_ylabel("Count")                                                                                                                                                         
plt.tight_layout()          

In [ ]:
pdd_filtered.exp_id_sampled

In [ ]:
debm.surface_melt_flux.mean(dim="time").isel(exp_id=0).plot()

In [ ]:
df = pdd.pism_config.to_series().apply(json.loads).apply(pd.Series)[pdd_uq_vars]

In [ ]:
import pandas as pd

In [ ]:
df